In [ ]:
# Setup
#pip install -q openai python-dotenv


## 1) Import


In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI


## 2) Load environment variables - please read instructions carefully


In [ ]:
# if you are running in local, uncomment below line. also make sure you shall have a .env file
load_dotenv()

In [ ]:
# if you are running in google colab, uncomment below line. and replace "Your_API_Key" with your own openAI API key
# os.environ["OPENAI_API_KEY"] = "Your_API_Key"

In [ ]:
llm_name = "gpt-4"
openai_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=openai_key)


## 3) Tool


In [ ]:
# Implement the functions actions
def estimate_trip_cost(
    days: int,
    travelers: int,
    comfort: str = "mid",
):
    """
    Estimate a rough trip budget (SGD) using simple heuristics.
    comfort: budget | mid | premium
    Returns a breakdown and total estimate in SGD.
    """
    if days <= 0 or travelers <= 0:
        raise ValueError("days and travelers must be > 0")

    comfort = comfort.lower().strip()
    if comfort not in {"budget", "mid", "premium"}:
        raise ValueError("comfort must be one of: budget, mid, premium")

    # Very rough per-person-per-day estimates (SGD) excluding flights
    lodging_per_person_day = {"budget": 60, "mid": 140, "premium": 300}[comfort]
    food_per_person_day = {"budget": 30, "mid": 60, "premium": 120}[comfort]
    local_transport_per_person_day = {"budget": 10, "mid": 20, "premium": 50}[comfort]
    activities_per_person_day = {"budget": 20, "mid": 50, "premium": 120}[comfort]

    lodging = lodging_per_person_day * travelers * days
    food = food_per_person_day * travelers * days
    transport = local_transport_per_person_day * travelers * days
    activities = activities_per_person_day * travelers * days

    subtotal = lodging + food + transport + activities
    contingency = round(subtotal * 0.12)  # 12% buffer
    total = subtotal + contingency

    return f"total_estimate: {total}"

known_actions = {"estimate_trip_cost": estimate_trip_cost}


## 4) custom Agent


In [ ]:
# Create a agent
class Agent:
    def __init__(self, system=""):
        self.system = system
        self.messages = []
        if system:
            self.messages.append({"role": "system", "content": system})

    def __call__(self, message):
        self.messages.append({"role": "user", "content": message})
        result = self.execute()
        self.messages.append({"role": "assistant", "content": result})
        return result

    def execute(self):
        response = client.chat.completions.create(
            model=llm_name,
            temperature=0.4,
            messages=self.messages,
            store=True
        )
        return response.choices[0].message.content


In [ ]:
prompt = """
You run in a loop of Thought, Action, Observation.
At the end of the loop you output an Answer.
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you.
Observation will be the result of running those actions.

Important notes:
Do not invent any facts, numbers. Use available actions instead.

Your available actions are:

estimate_trip_cost:
e.g. estimate_trip_cost: 2, 2, "mid" // for 2 days, 2 travellers, mid comfort.
returns the estimated cost of a trip


""".strip()

In [ ]:
# Create the agent
agent = Agent(system=prompt)


## 5) Simple Query


In [ ]:
response = agent("a 2-day Tokyo trip for 2 adults. Mid comfort. How much roughly will it cost?")
print(response)

In [ ]:
#response = estimate_trip_cost(2, 2, "mid")

In [ ]:
#next_message = f"Observation: {response}"
#print(next_message)

In [ ]:
#response = agent(next_message)
#print(response)

In [ ]:
#print(f" Messages--> {agent.messages}")


## 6) Complex query


In [ ]:
question = "Which ones cost lesser, out of these 2 choices: '2-day Tokyo trip for 2 adults mid comfort' and '3-day malaysia trip for 2 adults premium comfort'?"
response = agent(question)
print(response)

In [ ]:
#next_prompt = "Observation: {}".format(estimate_trip_cost(2, 2, "mid"))
#print(next_prompt)

call the agent again with the next prompt

In [ ]:
#res = agent(next_prompt)
#print(res)

In [ ]:
#next_prompt = "Observation: {}".format(estimate_trip_cost(3, 2, "premium"))
#print(next_prompt)

call the agent again with the next prompt

In [ ]:
#res = agent(next_prompt)
#print(res)


## 7) Create a loop to automate the agent until the agent returns an answer


In [ ]:
import re

action_re = re.compile(r"^Action: (\w+): (.*)$")
param_re = re.compile(
    r"^\s*(\d+)\s*,\s*(\d+)\s*,\s*\"?(budget|mid|premium)\"?\s*$",
    re.IGNORECASE
)

# Create a query function
def query(question, max_turns=10):
    i = 0
    bot = Agent(prompt)
    next_prompt = question
    while i < max_turns:
        i += 1
        result = bot(next_prompt)
        print(result)
        actions = [action_re.match(a) for a in result.split("\n") if action_re.match(a)]
        if actions:
            # There is an action to run
            action, action_input = actions[0].groups()
            if action == "estimate_trip_cost":
                match = param_re.match(action_input)
                if not match:
                    raise Exception(f"Invalid parameters for estimate_trip_cost: {action_input}")
                days = int(match.group(1))
                travelers = int(match.group(2))
                comfort = match.group(3).lower()
                print(" -- running {} {}".format(action, action_input))
                observation = known_actions[action](days, travelers, comfort)
            else:
                observation = known_actions[action](action_input)
            print("Observation:", observation)
            next_prompt = "Observation: {}".format(observation)
        else:
            return

question = "Which ones cost lesser, out of these 2 choices: '2-day Tokyo trip for 2 adults mid comfort' and '3-day malaysia trip for 2 adults premium comfort'?"
query(question)